<a href="https://colab.research.google.com/github/Rancor06/Edutrack---Student-Performance-Dropout-Risk-Predictor-Innolift-Internship-Project-/blob/main/Notebook/Phase_2_Model_Building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import pickle
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, sep=";")
print("Dataset shape:", df.shape)

Saving predict+students+dropout+and+academic+success.zip to predict+students+dropout+and+academic+success.zip
Dataset shape: (4424, 37)


In [2]:
# ---------------------------------------------------------
# 2. Display first few records
# ---------------------------------------------------------
print("First 5 records:")
half = len(df.columns) // 2

print("First half of columns:")
display(df[df.columns[:half]].head())

print("\nSecond half of columns:")
display(df[df.columns[half:]].head())

First 5 records:
First half of columns:


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0



Second half of columns:


,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [3]:
# ---------------------------------------------------------
# 3. Check data types
# ---------------------------------------------------------
print("\nData types:")
print(df.dtypes.value_counts())


Data types:
int64      29
float64     7
object      1
Name: count, dtype: int64


In [4]:
# ---------------------------------------------------------
# 4. Check for missing values
# ---------------------------------------------------------
print("\nMissing values check:")
print(df.isnull().sum().sum())


Missing values check:
0


In [5]:
# ---------------------------------------------------------
# 5. Target distribution (worth checking early given imbalance)
# ---------------------------------------------------------
print("\nTarget distribution:")
print(df["Target"].value_counts())
print(df["Target"].value_counts(normalize=True) * 100)


Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64
Target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64


In [6]:
# ---------------------------------------------------------
# 6. Preprocessing
# ---------------------------------------------------------
X = df.drop("Target", axis=1)
y = df["Target"]

y = y.map({"Dropout": 0, "Enrolled": 1, "Graduate": 2})

In [7]:
# ---------------------------------------------------------
# 7. Train/test Split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (3539, 36)
Test shape: (885, 36)


In [8]:
# ---------------------------------------------------------
# 8. Train The Model
# ---------------------------------------------------------
model = DecisionTreeClassifier(
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)
model.fit(X_train, y_train)
print("Model Trained Successfully")

Model Trained Successfully


In [9]:
# ---------------------------------------------------------
# 9. Evaluate
# ---------------------------------------------------------
y_pred = model.predict(X_test)

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", accuracy_score(y_test, y_pred))

target_names = ["Dropout", "Enrolled", "Graduate"]
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=target_names))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Train accuracy: 0.8118112461147217
Test accuracy: 0.6700564971751413

Classification Report:
               precision    recall  f1-score   support

     Dropout       0.80      0.62      0.69       284
    Enrolled       0.37      0.62      0.46       159
    Graduate       0.81      0.72      0.76       442

    accuracy                           0.67       885
   macro avg       0.66      0.65      0.64       885
weighted avg       0.73      0.67      0.69       885


Confusion Matrix:
 [[175  75  34]
 [ 19  99  41]
 [ 26  97 319]]


In [10]:
# ---------------------------------------------------------
# 10. Feature Importance
# ---------------------------------------------------------
importances = pd.Series(model.feature_importances_, index=X.columns)
print("Top 10 most important features:")
print(importances.sort_values(ascending=False).head(10))

Top 10 most important features:
Curricular units 2nd sem (approved)       0.498790
Tuition fees up to date                   0.076710
Curricular units 1st sem (evaluations)    0.053071
Curricular units 1st sem (enrolled)       0.041053
Curricular units 2nd sem (grade)          0.034962
Mother's occupation                       0.027863
Previous qualification (grade)            0.026998
Age at enrollment                         0.020832
Curricular units 2nd sem (enrolled)       0.020277
Course                                    0.019832
dtype: float64


In [11]:
# ---------------------------------------------------------
# 11. Save Trained Model as .pkl
# ---------------------------------------------------------
with open("dropout_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Decision Tree model saved to dropout_model.pkl")
files.download("dropout_model.pkl")

Decision Tree model saved to dropout_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>